## Limpeza, padronização e transformação dos CSV "Mortalidade_Geral"

In [1]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path
import warnings

# Suprimir warnings
warnings.filterwarnings('ignore')

# Configurar caminhos
RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(exist_ok=True, parents=True)


COLUNAS_INTERESSE = ['CODMUNRES', 'CAUSABAS', 'DTOBITO']

print("Ambiente configurado.")

Ambiente configurado.


Como o arquivo é muito grande, essa função lê o arquivo de mortalidade em pedaços de 100.000 linhas, filtra o que é cardio e dps agrega, para não correr o risco de maquina fracas como a minha não consiga terminar o processo. 

In [4]:
def processar_mortalidade(ano):
    print(f'\n=== Processando Mortalidade {ano} ===')
    
    # CORREÇÃO DO NOME: Padrão exato da sua imagem (Maiúsculas e Underline)
    file_name = f'Mortalidade_Geral_{ano}.csv'
    file_path = RAW_PATH / file_name
    
    if not file_path.exists():
        print(f'[ERRO] Arquivo nao encontrado: {file_path}')
        return pd.DataFrame()

    print(f'   Lendo arquivo: {file_name} ...')

    chunks = []
    
    # Leitura Otimizada (Chunks de 50k linhas)
    try:
        iterator = pd.read_csv(
            file_path, 
            sep=';', 
            usecols=['CODMUNRES', 'CAUSABAS'], # Lê apenas o necessário
            encoding='latin1',
            dtype={'CODMUNRES': str, 'CAUSABAS': str}, # Força texto para economizar memória
            chunksize=50000,
            low_memory=False
        )
        
        contador = 0
        for chunk in iterator:
            contador += 1
            if contador % 50 == 0: print(f'      ...lote {contador} processado...')
            
            # 1. Filtra Doenças Cardiovasculares (Começam com 'I')
            chunk = chunk.dropna(subset=['CAUSABAS'])
            chunk_cardio = chunk[chunk['CAUSABAS'].str.startswith('I', na=False)].copy()
            
            if not chunk_cardio.empty:
                # 2. Padroniza Município (6 dígitos)
                chunk_cardio['codmun'] = chunk_cardio['CODMUNRES'].str.slice(0, 6)
                
                # 3. Conta (Agrega imediatamente)
                resumo = chunk_cardio.groupby('codmun').size().reset_index(name='obitos')
                chunks.append(resumo)
                
    except Exception as e:
        print(f'[ERRO] Falha crítica na leitura: {e}')
        return pd.DataFrame()

    # Consolidação Final do Ano
    if chunks:
        df_ano = pd.concat(chunks, ignore_index=True)
        # Soma os totais de todos os pedacinhos
        df_ano = df_ano.groupby('codmun')['obitos'].sum().reset_index()
        df_ano.rename(columns={'obitos': 'mortes_cardio'}, inplace=True)
        df_ano['ano'] = int(ano)
        
        print(f'[SUCESSO] Ano {ano} finalizado. Total de mortes cardio: {df_ano["mortes_cardio"].sum():,}')
        return df_ano
    else:
        print(f'[AVISO] Nenhuma morte cardiovascular encontrada em {ano}.')
        return pd.DataFrame()

# --- Execução ---
dfs_mort = []
for ano in [2007, 2010, 2015]:
    df = processar_mortalidade(ano)
    if not df.empty:
        dfs_mort.append(df)

if dfs_mort:
    df_final = pd.concat(dfs_mort, ignore_index=True)
    # Filtra códigos inválidos
    df_final = df_final[df_final['codmun'].str.len() == 6]
    
    # Salva
    output_file = PROCESSED_PATH / 'mortalidade_cardio_clean.parquet'
    df_final.to_parquet(output_file, index=False)
    print(f'\n[CONCLUÍDO] Arquivo salvo em: {output_file}')
    print(df_final.head())


=== Processando Mortalidade 2007 ===
   Lendo arquivo: Mortalidade_Geral_2007.csv ...
[SUCESSO] Ano 2007 finalizado. Total de mortes cardio: 308,466

=== Processando Mortalidade 2010 ===
   Lendo arquivo: Mortalidade_Geral_2010.csv ...
[SUCESSO] Ano 2010 finalizado. Total de mortes cardio: 326,371

=== Processando Mortalidade 2015 ===
   Lendo arquivo: Mortalidade_Geral_2015.csv ...
[SUCESSO] Ano 2015 finalizado. Total de mortes cardio: 349,642

[CONCLUÍDO] Arquivo salvo em: ../data/processed/mortalidade_cardio_clean.parquet
   codmun  mortes_cardio   ano
0  110000              2  2007
1  110001             21  2007
2  110002             69  2007
3  110003              6  2007
4  110004             70  2007


Consolidação 

In [5]:
dfs_mortalidade = []

# Processar os anos definidos
for ano in [2007, 2010, 2015]:
    df_temp = processar_mortalidade(ano)
    if not df_temp.empty:
        dfs_mortalidade.append(df_temp)

# Juntar tudo
if dfs_mortalidade:
    df_final_mort = pd.concat(dfs_mortalidade, ignore_index=True)
    
    # Filtrar codigos de municipio invalidos (tamanho diferente de 6)
    df_final_mort = df_final_mort[df_final_mort['codmun'].str.len() == 6]
    
    print('\n=== Dataset Mortalidade Consolidado ===')
    print(f'Shape: {df_final_mort.shape}')
    print(df_final_mort.head())
    
    # Salvar Parquet
    output_file = PROCESSED_PATH / 'mortalidade_cardio_clean.parquet'
    df_final_mort.to_parquet(output_file, index=False)
    print(f'[SUCESSO] Salvo em: {output_file}')
else:
    print('[ERRO] Nenhum dado processado.')


=== Processando Mortalidade 2007 ===
   Lendo arquivo: Mortalidade_Geral_2007.csv ...
[SUCESSO] Ano 2007 finalizado. Total de mortes cardio: 308,466

=== Processando Mortalidade 2010 ===
   Lendo arquivo: Mortalidade_Geral_2010.csv ...
[SUCESSO] Ano 2010 finalizado. Total de mortes cardio: 326,371

=== Processando Mortalidade 2015 ===
   Lendo arquivo: Mortalidade_Geral_2015.csv ...
[SUCESSO] Ano 2015 finalizado. Total de mortes cardio: 349,642

=== Dataset Mortalidade Consolidado ===
Shape: (16674, 3)
   codmun  mortes_cardio   ano
0  110000              2  2007
1  110001             21  2007
2  110002             69  2007
3  110003              6  2007
4  110004             70  2007
[SUCESSO] Salvo em: ../data/processed/mortalidade_cardio_clean.parquet


SQL só para visualizar se está tudo nos conformes kkkk 

In [6]:
print('\n=== Validacao com DuckDB ===')

con = duckdb.connect()
con.register('tb_mortalidade', df_final_mort)

# 1. Resumo por Ano (Verificar se tem dados para todos os anos)
query_resumo = """
SELECT 
    ano, 
    COUNT(*) as qtd_municipios_com_obito,
    SUM(mortes_cardio) as total_obitos_brasil,
    AVG(mortes_cardio)::INT as media_obitos_cidade,
    MAX(mortes_cardio) as max_obitos_cidade
FROM tb_mortalidade
GROUP BY ano
ORDER BY ano
"""
print('\n[SQL] Resumo de Mortalidade Cardiovascular:')
print(con.execute(query_resumo).df())

# 2. Checagem Top 5 Cidades (Validacao de consistencia - SP e Rio devem liderar)
query_top = """
SELECT codmun, mortes_cardio
FROM tb_mortalidade
WHERE ano = 2015
ORDER BY mortes_cardio DESC
LIMIT 5
"""
print('\n[SQL] Top 5 Cidades com mais obitos cardio (2015):')
print(con.execute(query_top).df())


=== Validacao com DuckDB ===

[SQL] Resumo de Mortalidade Cardiovascular:
    ano  qtd_municipios_com_obito  total_obitos_brasil  media_obitos_cidade  \
0  2007                      5548             308466.0                   56   
1  2010                      5555             326371.0                   59   
2  2015                      5571             349642.0                   63   

   max_obitos_cidade  
0              21778  
1              22822  
2              23982  

[SQL] Top 5 Cidades com mais obitos cardio (2015):
   codmun  mortes_cardio
0  355030          23982
1  330455          15812
2  292740           3771
3  310620           3632
4  230440           3435
